# Inspect a tiny forward pass on a T4

One Modal call returns **ordinary dictionaries, lists, numbers, and strings**. The live profiler stays inside the remote function; no pickling of `prof` is needed.

Random inputs have shape `[batch_size, 1]` and the model produces two scores per example. Your current experiment performs `total_steps` warmup forward passes, then profiles **one forward pass** on the last batch. The loss, backward, and optimizer calls remain commented out in the measured block.

Shape recording, memory tracking, source stacks, and FLOP estimation are enabled. These options add overhead; use this capture to inspect the work rather than as a clean latency measurement.

In [1]:
import modal

app = modal.App("toy-pytorch-profiler")
image = modal.Image.debian_slim(python_version="3.12").pip_install("torch")

In [ ]:
@app.function(image=image, gpu="T4", timeout=300, serialized=True)
def train_and_profile(batch_size: int = 10, total_steps: int = 10) -> dict:
    import os
    import tempfile
    from pathlib import Path

    import torch
    from torch import nn
    from torch.profiler import ProfilerActivity, profile
    from torch.utils.data import DataLoader, TensorDataset

    torch.manual_seed(0)
    device = torch.device("cuda")
    d_in, d_h, d_out = 1, 5, 2

    # Make exactly enough random examples for total_steps full batches.
    dataset = TensorDataset(
        torch.randn(total_steps * batch_size, d_in),
        torch.randint(d_out, (total_steps * batch_size,)),
    )
    loader = DataLoader(dataset, batch_size=batch_size, num_workers=0)

    model = nn.Sequential(
        nn.Linear(d_in, d_out),
        # nn.Linear(d_h, d_h),
        # nn.Linear(d_h, d_out),
    ).to(device)
    # optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
    # loss_function = nn.CrossEntropyLoss()

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        model(x)
    torch.cuda.synchronize()  # Finish warmup before starting the capture.
    with profile(
        activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
        record_shapes=True,
        profile_memory=True,
        with_stack=True,
        with_flops=True,
        # Retain source stacks in exported events on this PyTorch build.
        experimental_config=torch._C._profiler._ExperimentalConfig(verbose=True),
    ) as prof:
        model(x)
        # optimizer.zero_grad()
        # logits = model(x)
        # loss = loss_function(logits, y)
        # loss.backward()
        # optimizer.step()
        torch.cuda.synchronize()  # Include completion of the forward pass.

    # Copy useful fields into ordinary values, while the profiler is local.
    def as_row(event):
        return {
            "name": event.key,
            "count": event.count,
            "input_shapes": event.input_shapes,
            "stack": list(event.stack or []),
            "flops": event.flops,
            "cpu_time_us": event.cpu_time_total,
            "self_cpu_time_us": event.self_cpu_time_total,
            "device_time_us": event.device_time_total,
            "self_device_time_us": event.self_device_time_total,
            "cpu_memory_bytes": event.cpu_memory_usage,
            "self_cpu_memory_bytes": event.self_cpu_memory_usage,
            "device_memory_bytes": event.device_memory_usage,
            "self_device_memory_bytes": event.self_device_memory_usage,
        }

    # Preserve separate averages for different input shapes.
    averages = prof.key_averages(group_by_input_shape=True)

    # prof.events() has already linked every event to its parent. `event.id` is the CUPTI
    # correlation id, which is 0 for every python_function, so it cannot key that link:
    # each row carries its own position instead, and `parent` is the position of its
    # cpu_parent. `event.kernels` is PyTorch's own operator-to-kernel association.
    ordered = list(prof.events())
    position = {id(event): i for i, event in enumerate(ordered)}
    events = [
        {
            **as_row(event),
            "index": i,
            "parent": position.get(id(event.cpu_parent)) if event.cpu_parent else None,
            "kernels": [
                {"name": k.name, "device": k.device, "duration_us": k.duration}
                for k in event.kernels
            ],
            "thread": event.thread,
            "device_index": event.device_index,
            "device_type": str(event.device_type),
            "correlation_id": event.id,
            "sequence_nr": event.sequence_nr,
            "start_us": event.time_range.start,
            "end_us": event.time_range.end,
        }
        for i, event in enumerate(ordered)
    ]
    # The full trace preserves timestamps, kernel links, and input type metadata.
    with tempfile.TemporaryDirectory() as temp:
        path = Path(temp) / "trace.json"
        prof.export_chrome_trace(str(path))
        trace_json = path.read_text()

    return {
        # Correlation ids, External ids and pids all restart per container, so every
        # trace has to be read against the worker that produced it.
        "worker": os.environ.get("MODAL_TASK_ID", "local"),
        "pid": os.getpid(),
        "table": averages.table(sort_by="self_device_time_total", row_limit=-1),
        "key_averages": [as_row(event) for event in averages],
        "events": events,
        "trace_json": trace_json,
    }

In [3]:
batch_size = 10
total_steps = 10

with app.run():
    results = train_and_profile.remote(batch_size=batch_size, total_steps=total_steps)

print(results["table"])

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ----------------------------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls                        Input Shapes   Total FLOPs  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ----------------------------------  ------------  
                                            aten::addmm         8.17%     203.934us   

The returned dictionary contains:

| Key | Inspect in Jupyter |
| --- | --- |
| `worker` | Modal container id (`MODAL_TASK_ID`). |
| `pid` | The container's `os.getpid()`. With `worker` and a row's `thread`, this identifies a thread. |
| `table` | The formatted operator summary, printed above. |
| `key_averages` | Aggregate rows grouped by operator name **and input shapes**. |
| `events` | Individual events, each with its position, its parent's position, and its kernels. |
| `trace_json` | Full Chrome trace as JSON text, including kernel launch geometry and input dtype metadata. |

Times in the dictionaries are **microseconds**. Memory fields are **net allocation deltas in bytes**, not peak usage, and can be negative. FLOPs are estimated arithmetic operation counts for supported operations such as matrix multiplication; zero or missing values do not prove that an operation did no arithmetic. Stack lists can be empty for some events or builds. The individual events preserve available stacks; the averages above group by shape, not stack.

`record_shapes=True` records input dimensions. Dtype information, when emitted, is available under trace event `args["Input type"]`; it is not a universal field on averaged events. Shapes such as `[]` can represent scalar or non-tensor arguments.

Operator rows and their kernel rows can describe the same GPU work. Do not sum the rows or interpret their percentages as GPU utilization.

Each `events` row carries `index` (its own position), `parent` (the position of its `cpu_parent`, or `None` for a root), and `kernels` (the kernels PyTorch associated with it). `correlation_id` is `event.id`, the CUPTI correlation id; it is `0` for every `python_function` and so cannot identify a row. The tree at the bottom of this notebook joins `parent` to `index`.

References: [PyTorch profiler](https://docs.pytorch.org/docs/stable/profiler.html) · [Modal in Jupyter](https://modal.com/docs/guide/jupyter-notebooks)

The capture uses PyTorch's experimental `verbose=True` setting to retain stacks in exported events on this build, as in the [official profiler recipe](https://github.com/pytorch/tutorials/blob/main/recipes_source/recipes/profiler_recipe.py). This setting is version-dependent.

In [4]:
# Inspect one matrix operation, including its input shapes and FLOP estimate.
next(row for row in results["key_averages"] if row["name"] == "aten::addmm")

{'name': 'aten::addmm',
 'count': 1,
 'input_shapes': [[2], [10, 1], [1, 2], [], []],
 'stack': [],
 'flops': 40,
 'cpu_time_us': 2382.618,
 'self_cpu_time_us': 203.93399999999974,
 'device_time_us': 8.96499999999969,
 'self_device_time_us': 8.96499999999969,
 'cpu_memory_bytes': 0,
 'self_cpu_memory_bytes': 0,
 'device_memory_bytes': 512,
 'self_device_memory_bytes': 512}

In [5]:
# Individual events retain available Python source stacks.
stacked_events = [event for event in results["events"] if event["stack"]]
stacked_events[0] if stacked_events else "This capture did not expose Python stacks in prof.events(); inspect the full trace below."

{'name': 'threading.py(1075): _bootstrap_inner',
 'count': 1,
 'input_shapes': [],
 'stack': ['threading.py(1032): _bootstrap'],
 'flops': 0,
 'cpu_time_us': 3325.844,
 'self_cpu_time_us': 2.4420000000000073,
 'device_time_us': 0,
 'self_device_time_us': 0,
 'cpu_memory_bytes': 0,
 'self_cpu_memory_bytes': 0,
 'device_memory_bytes': 0,
 'self_device_memory_bytes': 0,
 'index': 1,
 'parent': 0,
 'kernels': [],
 'thread': 18446744073709551615,
 'device_index': 0,
 'device_type': 'DeviceType.CPU',
 'correlation_id': 0,
 'sequence_nr': -1,
 'start_us': 147.045,
 'end_us': 3472.889}

In [6]:
import json

# This also verifies that the response contains only JSON-serializable data.
json.dumps(results, allow_nan=False)
trace = json.loads(results["trace_json"])
next(
    (event for event in trace["traceEvents"]
     if event.get("name") == "aten::addmm" and event.get("cat") == "cpu_op"),
    None,
)

{'ph': 'X',
 'cat': 'cpu_op',
 'name': 'aten::addmm',
 'pid': 2,
 'tid': 2,
 'ts': 7408453105222.385,
 'dur': 2382.618,
 'args': {'External id': 5,
  'Sequence number': 21,
  'Record function id': 0,
  'Concrete Inputs': ['', '', '', '1', '1'],
  'Input type': ['float', 'float', 'float', 'Scalar', 'Scalar'],
  'Input Strides': [[1], [1, 1], [1, 1], [], []],
  'Fwd thread id': 0,
  'Call stack': '<built-in function linear>;torch/nn/modules/linear.py(130): forward;torch/nn/modules/module.py(1787): _call_impl;nn.Module: Linear_0;torch/nn/modules/container.py(254): forward;torch/nn/modules/module.py(1787): _call_impl;nn.Module: Sequential_0;/var/folders/_9/c3pgt8bn23n2jqgj_1cfghrm0000gn/T/ipykernel_13904/2840277697.py(35): train_and_profile;modal/_runtime/container_io_manager.py(265): call_function_sync;modal/_container_entrypoint.py(187): run_input_sync;modal/_container_entrypoint.py(262): call_function;modal/_container_entrypoint.py(465): run_function;modal/_container_entrypoint.py(472):

To open the complete timeline in a Chrome-trace-compatible viewer, save it locally:

```python
from pathlib import Path
Path("toy.trace.json").write_text(results["trace_json"], encoding="utf-8")
```

You can filter `results["events"]` and regroup the returned rows in Python without another GPU call. Profiler methods themselves, such as changing `key_averages()` grouping, must run inside the Modal function before it returns; the exported data is not a live profiler.

In [7]:
from pathlib import Path
Path("toy.trace.json").write_text(results["trace_json"], encoding="utf-8")

184732

In [8]:
import pandas as pd

In [9]:
df = pd.DataFrame(results['key_averages'])

In [10]:
df

,name,count,input_shapes,stack,flops,cpu_time_us,self_cpu_time_us,device_time_us,self_device_time_us,cpu_memory_bytes,self_cpu_memory_bytes,device_memory_bytes,self_device_memory_bytes
0,aten::linear,1,"[[10, 1], [2, 1], [2]]",[],0,2477.964,36.092,8.965,0.000,0,0,512,0
1,aten::t,1,"[[2, 1]]",[],0,59.254,32.745,0.000,0.000,0,0,0,0
2,aten::transpose,1,"[[2, 1], [], []]",[],0,26.509,18.778,0.000,0.000,0,0,0,0
3,aten::as_strided,1,"[[2, 1], [], [], []]",[],0,7.731,7.731,0.000,0.000,0,0,0,0
4,aten::addmm,1,"[[2], [10, 1], [1, 2], [], []]",[],40,2382.618,203.934,8.965,8.965,0,0,512,512
5,aten::expand,1,"[[2], [], []]",[],0,8.137,6.216,0.000,0.000,0,0,0,0
6,aten::as_strided,1,"[[2], [], [], []]",[],0,1.921,1.921,0.000,0.000,0,0,0,0
7,Activity Buffer Request,1,[],[],0,2092.888,2092.888,0.000,0.000,0,0,0,0
8,cudaLaunchKernel,2,[],[],0,77.659,77.659,0.000,0.000,0,0,0,0
9,"void at::native::elementwise_kernel<128, 2, at...",1,[],[],0,0.000,0.000,4.322,4.322,0,0,0,0


In [11]:
df_events = pd.DataFrame(results['events'])

In [12]:
results['events'][50]

{'name': 'nn.Module: Linear_0',
 'count': 1,
 'input_shapes': [],
 'stack': ['torch/nn/modules/container.py(254): forward',
  'torch/nn/modules/module.py(1787): _call_impl',
  'nn.Module: Sequential_0',
  '/var/folders/_9/c3pgt8bn23n2jqgj_1cfghrm0000gn/T/ipykernel_13904/2840277697.py(35): train_and_profile',
  'modal/_runtime/container_io_manager.py(265): call_function_sync',
  'modal/_container_entrypoint.py(187): run_input_sync',
  'modal/_container_entrypoint.py(262): call_function',
  'modal/_container_entrypoint.py(465): run_function',
  'modal/_container_entrypoint.py(472): main',
  'modal/_container_entrypoint.py(490): <module>',
  '<frozen runpy>(88): _run_code',
  '<frozen runpy>(198): _run_module_as_main'],
 'flops': 0,
 'cpu_time_us': 2592.265,
 'self_cpu_time_us': 5.802999999999884,
 'device_time_us': 8.96499999999969,
 'self_device_time_us': 0.0,
 'cpu_memory_bytes': 0,
 'self_cpu_memory_bytes': 0,
 'device_memory_bytes': 512,
 'self_device_memory_bytes': 0,
 'index': 50,


## The operator tree

`prof.events()` has already linked every event to its parent, so the tree is rebuilt from
those links rather than re-derived. `op_tree.from_events` reads `results["events"]`, joins
each row to the row at its `parent` index, and hangs each operator's `kernels` underneath it.

The rows no longer carry `event.id`. That field is the CUPTI correlation id, `0` for every
`python_function`, so it cannot key the parent link; row position does. Kernels come from
`event.kernels`, which is the association PyTorch itself made from the linked correlation id,
so no correlation matching happens on this side either.

Identity is `(worker, pid, tid)`: the Modal container id, the container's `os.getpid()`, and
`event.thread`. Correlation ids, external ids and pids all restart in every container, so each
worker's rows are built on their own and the roots are concatenated afterward.

`op_tree.build` is the other entry point. It reads a `trace_json`, derives nesting from time
containment, and is what to use for a trace file with no Python objects behind it, or when you
need what only the trace records: kernel start times, grid and block dimensions, occupancy.
The two agree on structure because PyTorch derives `cpu_parent` by containment as well.

In [13]:
from op_tree import from_events, render

# One dict per worker. `train_and_profile.map(...)` already returns that list.
runs = [results]

roots = [op for run in runs for op in from_events(run["events"], run["worker"], run["pid"])]
nodes = [op for root in roots for _, op in root.walk()]

print(f"{len(runs)} worker(s), {len(roots)} roots, {len(nodes)} nodes")
print(sorted({op.thread for op in nodes}))

1 worker(s), 2 roots, 168 nodes
[('ta-01M3DBTC1F05V201SJXYSN2G5R', 0, -1), ('ta-01M3DBTC1F05V201SJXYSN2G5R', 2, 1), ('ta-01M3DBTC1F05V201SJXYSN2G5R', 2, 18446744073709551615)]


In [14]:
# The roots are the interpreter entry points, so render from the model call down instead.
top = next((op for op in nodes if op.name.startswith("nn.Module:")), roots[0])
print(render([top]))

nn.Module: Sequential_0  +0.0 2680.9us (self 11.7) [cpu_op]
  torch/nn/modules/module.py(1787): _call_impl  +9.6 2669.3us (self 15.1) [cpu_op]
    <built-in method _get_tracing_state of pybind11_builtins.pybind11_detail_function_record_v1_system_libstdcpp_gxx_abi_1xxx_use_cxx11_abi_1 object at 0x2a5afa040f10>  +13.7 2.4us (self 2.4) [cpu_op]
    torch/nn/modules/container.py(254): forward  +26.2 2651.7us (self 50.8) [cpu_op]
      torch/nn/modules/container.py(246): __iter__  +29.2 8.6us (self 6.8) [cpu_op]
        <built-in method values of dict object at 0x2a5bb91a9780>  +33.9 1.1us (self 1.1) [cpu_op]
        <built-in function iter>  +37.0 0.7us (self 0.7) [cpu_op]
      nn.Module: Linear_0  +82.6 2592.3us (self 5.8) [cpu_op]
        torch/nn/modules/module.py(1787): _call_impl  +86.3 2586.5us (self 8.8) [cpu_op]
          <built-in method _get_tracing_state of pybind11_builtins.pybind11_detail_function_record_v1_system_libstdcpp_gxx_abi_1xxx_use_cxx11_abi_1 object at 0x2a5afa040f1

In [15]:
# The operators that launch kernels themselves. `device_time` is a subtree sum, so
# filtering on it would also report every ancestor of a launch.
for op in nodes:
    kernels = [c for c in op.children if c.cat == "kernel"]
    if not kernels:
        continue
    launch = sum(c.dur for c in op.children if c.cat == "cuda_runtime")
    print(f"{op.worker} pid={op.pid} tid={op.tid}  {op.name} {op.shapes}")
    print(f"    under {' <- '.join(a.short for a in op.ancestors())}")
    print(f"    called from {op.stack[0] if op.stack else '(no stack)'}")
    print(f"    {sum(k.dur for k in kernels):.1f}us device against {launch:.1f}us of launches")
    for kernel in kernels:
        print(f"    -> {kernel.short}  {kernel.dur:.1f}us")

ta-01M3DBTC1F05V201SJXYSN2G5R pid=2 tid=1  aten::addmm [[2], [10, 1], [1, 2], [], []]
    under aten::linear <- <built-in function linear> <- torch/nn/modules/linear.py(130): forward <- torch/nn/modules/module.py(1787): _call_impl <- nn.Module: Linear_0 <- torch/nn/modules/container.py(254): forward <- torch/nn/modules/module.py(1787): _call_impl <- nn.Module: Sequential_0 <- /var/folders/_9/c3pgt8bn23n2jqgj_1cfghrm0000gn/T/ipykernel_13904/2840277697.py(35): train_and_profile <- modal/_runtime/container_io_manager.py(265): call_function_sync <- modal/_container_entrypoint.py(187): run_input_sync <- modal/_container_entrypoint.py(262): call_function <- modal/_container_entrypoint.py(465): run_function <- modal/_container_entrypoint.py(472): main <- modal/_container_entrypoint.py(490): <module> <- <frozen runpy>(88): _run_code <- <frozen runpy>(198): _run_module_as_main
    called from <built-in function linear>
    9.0us device against 77.7us of launches
    -> at::native::elementwise_k

In [17]:
# Every node as a row, keyed by (worker, pid, tid), for filtering and grouping.
df_nodes = pd.DataFrame(
    {
        "worker": op.worker,
        "pid": op.pid,
        "tid": op.tid,
        "cat": op.cat,
        "name": op.short,
        "depth": depth,
        "parent": op.parent.short if op.parent else None,
        "dur_us": op.dur,
        "self_us": op.self_time,
        "device_us": op.device_time,
        "shapes": op.shapes,
        "frame": op.stack[0] if op.stack else None,
    }
    for root in roots
    for depth, op in root.walk()
)
df_nodes

,worker,pid,tid,cat,name,depth,parent,dur_us,self_us,device_us,shapes,frame
0,ta-01M3DBTC1F05V201SJXYSN2G5R,2,18446744073709551615,cpu_op,threading.py(1032): _bootstrap,0,NaN,3328.550,2.706,0.0,[],NaN
1,ta-01M3DBTC1F05V201SJXYSN2G5R,2,18446744073709551615,cpu_op,threading.py(1075): _bootstrap_inner,1,threading.py(1032): _bootstrap,3325.844,2.442,0.0,[],threading.py(1032): _bootstrap
2,ta-01M3DBTC1F05V201SJXYSN2G5R,2,18446744073709551615,cpu_op,threading.py(1012): run,2,threading.py(1075): _bootstrap_inner,3323.402,1.831,0.0,[],threading.py(1075): _bootstrap_inner
3,ta-01M3DBTC1F05V201SJXYSN2G5R,2,18446744073709551615,cpu_op,synchronicity/synchronizer.py(230): thread_inner,3,threading.py(1012): run,3321.571,3.799,0.0,[],threading.py(1012): run
4,ta-01M3DBTC1F05V201SJXYSN2G5R,2,18446744073709551615,cpu_op,asyncio/runners.py(195): run,4,synchronicity/synchronizer.py(230): thread_inner,3317.772,2.560,0.0,[],synchronicity/synchronizer.py(230): thread_inner
...,...,...,...,...,...,...,...,...,...,...,...,...
163,ta-01M3DBTC1F05V201SJXYSN2G5R,2,1,cuda_runtime,cudaDeviceSynchronize,16,<built-in function _cuda_synchronize>,2.401,2.401,0.0,[],<built-in function _cuda_synchronize>
164,ta-01M3DBTC1F05V201SJXYSN2G5R,2,1,cpu_op,torch/cuda/__init__.py(719): __exit__,15,torch/cuda/__init__.py(1271): synchronize,16.908,6.895,0.0,[],torch/cuda/__init__.py(1271): synchronize
165,ta-01M3DBTC1F05V201SJXYSN2G5R,2,1,cpu_op,<built-in function _cuda_maybeExchangeDevice>,16,torch/cuda/__init__.py(719): __exit__,0.854,0.854,0.0,[],torch/cuda/__init__.py(719): __exit__
166,ta-01M3DBTC1F05V201SJXYSN2G5R,2,1,cpu_op,<built-in function perf_counter_ns>,16,torch/cuda/__init__.py(719): __exit__,0.952,0.952,0.0,[],torch/cuda/__init__.py(719): __exit__
